In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split,RandomizedSearchCV, cross_val_score,GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score , r2_score, mean_absolute_error, mean_squared_error
import joblib
import streamlit as st
from sklearn.neighbors import KNeighborsClassifier

In [2]:
df=pd.read_csv('patients_pro_max_with_priority.csv')

In [3]:
df['gender'] = df['gender'].replace({'Female': 0, 'Male': 1})

C:\Users\Administrateur\AppData\Local\Temp\ipykernel_8088\3590137998.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['gender'] = df['gender'].replace({'Female': 0, 'Male': 1})


In [4]:
df.head()

,patient_id,age,gender,heart_rate,respiratory_rate,oxygen_saturation,blood_pressure,pain_level,condition,needs_surgery,surgery_duration,bp_mean,priority
0,P001,67,0,80,18,96,108/82,2,10,True,73,90.666667,2
1,P002,27,1,81,22,93,139/75,3,10,True,77,96.333333,3
2,P003,51,1,104,15,94,137/97,3,9,True,54,110.333333,4
3,P004,69,1,62,16,97,96/80,2,8,True,47,85.333333,4
4,P005,48,0,81,23,91,117/84,3,7,True,76,95.000000,6


In [5]:
df.isnull().sum()

patient_id           0
age                  0
gender               0
heart_rate           0
respiratory_rate     0
oxygen_saturation    0
blood_pressure       0
pain_level           0
condition            0
needs_surgery        0
surgery_duration     0
bp_mean              0
priority             0
dtype: int64

In [6]:
df['needs_surgery']=df['needs_surgery'].replace({True:1,False:0})

C:\Users\Administrateur\AppData\Local\Temp\ipykernel_8088\1016333406.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['needs_surgery']=df['needs_surgery'].replace({True:1,False:0})


In [7]:
x=df[['age','gender','heart_rate','respiratory_rate','oxygen_saturation','pain_level','condition','bp_mean','needs_surgery']]
y=df['priority']

In [8]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
print('training feature shape',x_train.shape)
print('testing feature shape',x_test.shape)

training feature shape (12000, 9)
testing feature shape (3000, 9)


In [9]:
# LinearRegression

The history saving thread hit an unexpected error (OperationalError('database is locked')).History will not be written to the database.


In [10]:
lr_model = LinearRegression()
lr_model.fit(x_train, y_train)
y_pred = lr_model.predict(x_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Linear Regression Results:")
print(f"Test MSE: {mse:.4f}")
print(f"Test R2: {r2:.4f}")

Linear Regression Results:
Test MSE: 1.3436
Test R2: 0.8173


In [11]:
cv_scores = cross_val_score(lr_model, x_train, y_train, cv=5, scoring='r2')
print(f"Cross-Validation R2 Scores: {cv_scores}")
print(f"Mean CV R2: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

Cross-Validation R2 Scores: [0.81167384 0.81021199 0.80715888 0.80876883 0.82247311]
Mean CV R2: 0.8121 (+/- 0.0108)


### Random Forest Regressor

In [12]:
# Best Parameters: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}
# {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 600, 'random_state': 59}
 # {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 741, 'random_state': 91} max_features='log2',min_samples_leaf=1,min_samples_split=3
rf_model= RandomForestRegressor(n_estimators=100,max_depth=None,random_state=42)
rf_model.fit(x_train,y_train)
y_pred_rf=rf_model.predict(x_test)

In [13]:
print("R2 score:", r2_score(y_test, y_pred_rf))
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("MSE:", mean_squared_error(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))

R2 score: 0.9999608053019583
MAE: 0.00118333333333333
MSE: 0.00028816666666666693
RMSE: 0.016975472502014987


In [19]:
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats

param_dist = {
    'n_estimators': stats.randint(0, 1000),
    'max_depth': (None,5, 50,100,120,150),
    'min_samples_split': stats.randint(0, 20),
    'min_samples_leaf': stats.randint(0, 10),
    'max_features': ['auto', 'sqrt', 'log2'],
    'random_state' :stats.randint(0,100),
}

rand_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist,
    n_iter=50,  # number of random combinations
    cv=5,
    n_jobs=-1,
    scoring='r2',
    random_state=42,
    verbose=2
)

rand_search.fit(x_train, y_train)
print("Best Parameters:", rand_search.best_params_)


Fitting 5 folds for each of 50 candidates, totalling 250 fits


C:\Users\Administrateur\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
130 fits failed out of a total of 250.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
44 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Administrateur\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrateur\anaconda3\Lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\Administrateur\anaconda3\Lib\site-packages\sklearn\base.p

Best Parameters: {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 600, 'random_state': 59}


In [51]:
# Hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt']
}

# Grid search
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    cv=5,             # 5-fold cross-validation
    n_jobs=-1,        # use all cores
    scoring='r2',     # optimize for R²
    verbose=2
)

grid_search.fit(x_train, y_train)

# Best parameters
print("Best Parameters:", grid_search.best_params_)

# Predict and evaluate
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)

print("R²:", r2_score(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))


Fitting 5 folds for each of 216 candidates, totalling 1080 fits


C:\Users\Administrateur\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
540 fits failed out of a total of 1080.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
68 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Administrateur\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrateur\anaconda3\Lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\Administrateur\anaconda3\Lib\site-packages\sklearn\base.

Best Parameters: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 500}


NameError: name 'X_test' is not defined

In [38]:
import joblib

# Save the model
joblib.dump(rf_model, "surgery_priority_model.pkl")

['surgery_priority_model.pkl']

In [20]:


import joblib
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Load your model
model = joblib.load("surgery_priority_model.pkl")

# Define input shape (batch, features)
initial_type = [('input', FloatTensorType([None, 9]))]

# Convert to ONNX
onnx_model = convert_sklearn(model, initial_types=initial_type)

# Save
with open("surgery_priority_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())


In [22]:
!pip install tensorflow

In [25]:
!pip install onnx

  Using cached onnx-1.19.0-cp313-cp313-win_amd64.whl.metadata (7.2 kB)
Using cached onnx-1.19.0-cp313-cp313-win_amd64.whl (16.5 MB)


In [26]:
!pip install onnx2tf

In [27]:
onnx2tf -i surgery_priority_model.onnx -o tf_model

SyntaxError: invalid syntax (233528674.py, line 1)

In [24]:
import onnx
from onnx_tf.backend import prepare

onnx_model = onnx.load("surgery_priority_model.onnx")
tf_rep = prepare(onnx_model)
tf_rep.export_graph("onnx_saved_model")   # <-- creates saved_model.pb


ModuleNotFoundError: No module named 'onnx_tf'

In [23]:

import tensorflow as tf

model = tf.saved_model.load("onnx_saved_model")
converter = tf.lite.TFLiteConverter.from_saved_model("onnx_saved_model")
tflite_model = converter.convert()

with open("surgery_priority_model.tflite", "wb") as f:
    f.write(tflite_model)


OSError: SavedModel file does not exist at: onnx_saved_model\{saved_model.pbtxt|saved_model.pb}